# UnGPT v2.0: Static vs Dynamic Architecture

7-layer multi-model pipeline with:
- **Static Truth (SuperGrok)**: L0, L3, L5, L7 - Intent validation, sanity checks
- **Dynamic Execution (Claude)**: L1, L4, L6 - Framing, code execution, publishing
- **Infinite Loop (Gemini/GPT)**: L2 - Generative horsepower

## Flow
```
Voice → L0(Grok) → L1(Claude) → L2(Gemini↔GPT) → L3(Grok) → L4(Claude) → GitHub
                                                                    ↓
                                              L5(Grok) → L7(Voice) → Speaker
```

## Cell 1: Environment Setup

In [ ]:
# Install dependencies
!pip install -q anthropic httpx pyyaml rich

import os
import sys

# Add parent directory to path
sys.path.insert(0, os.path.dirname(os.getcwd()))

# Set API keys (use Colab secrets or environment variables)
os.environ["XAI_API_KEY"] = ""  # SuperGrok
os.environ["ANTHROPIC_API_KEY"] = ""  # Claude
os.environ["GOOGLE_API_KEY"] = ""  # Gemini
os.environ["OPENAI_API_KEY"] = ""  # GPT
os.environ["GITHUB_TOKEN"] = ""  # GitHub push

print("Environment ready")

## Cell 2: Load Configuration

In [ ]:
CONFIG = {
    "models": {
        "l0_grok": "grok-beta",
        "l1_claude": "claude-sonnet-4-20250514",
        "l2_gemini": "gemini-2.0-flash-exp",
        "l2_gpt": "gpt-4o",
        "l3_grok": "grok-beta",
        "l4_claude": "claude-opus-4-20250514",
        "l5_grok": "grok-beta",
        "l6_claude": "claude-sonnet-4-20250514",
        "l7_grok": "grok-beta",
    },
    "pipeline": {
        "HARD_CAP_CYCLES": 10,
        "convergence_signal": "NO_SIGNIFICANT_CHANGES",
        "crm_abort_threshold": 3,
        "crm_reloop_min": 4,
        "crm_reloop_max": 5,
        "cost_alert_threshold": 0.25,
        "always_pr_check": False,
    },
    "tools": {
        "l2b_default": ["google_search_retrieval", "code_execution"],
        "internal_keywords": ["PNKLN", "AtomicGrok", "ShadowTag", "Judge"],
    },
    "github": {"repo_name": "ShadowTag-v2/aiyou-fastapi-services", "branch_prefix": "research/"},
    "analytics": {"db_path": "analytics/crm_db.sqlite"},
}

API_KEYS = {
    "XAI_API_KEY": os.getenv("XAI_API_KEY", ""),
    "ANTHROPIC_API_KEY": os.getenv("ANTHROPIC_API_KEY", ""),
    "GOOGLE_API_KEY": os.getenv("GOOGLE_API_KEY", ""),
    "OPENAI_API_KEY": os.getenv("OPENAI_API_KEY", ""),
    "GITHUB_TOKEN": os.getenv("GITHUB_TOKEN", ""),
}

print("Configuration loaded")
print(f"Models: {list(CONFIG['models'].keys())}")

## Cell 3: UnGPT Orchestrator Class

In [ ]:
from dataclasses import dataclass
from datetime import datetime

from rich.console import Console

console = Console()


@dataclass
class PipelineResult:
    success: bool
    query_id: str
    crm_score: float = 0.0
    final_answer: str = ""
    voice_script: str = ""
    github_url: str = ""
    relooped: bool = False
    total_cost: float = 0.0
    total_cycles: int = 0
    abort_reason: str = ""
    error: str = ""


class UnGPTOrchestrator:
    def __init__(self, config, api_keys):
        self.config = config
        self.api_keys = api_keys
        self.costs = {}
        self.query_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.start_time = None

    def _track_cost(self, layer: str, cost: float):
        self.costs[layer] = cost
        total = sum(self.costs.values())
        if total > self.config["pipeline"]["cost_alert_threshold"]:
            console.print(f"[yellow]Cost Alert: ${total:.4f}[/yellow]")

    async def run(self, voice_input: str) -> PipelineResult:
        self.start_time = datetime.now()
        console.print(f"\n[cyan]Query ID:[/cyan] {self.query_id}")

        total_cycles = 0

        try:
            # === L0: SuperGrok Intake ===
            console.print("[yellow]→ L0: SuperGrok Intake[/yellow]")
            l0 = await self._l0_intake(voice_input)
            self._track_cost("L0", l0["cost"])
            normalized_query = l0["normalized"]
            spt1 = l0["spt1"]
            console.print("  ✓ Query normalized\n")

            # === L1: Claude Frame ===
            console.print("[yellow]→ L1: Claude Frame[/yellow]")
            l1 = await self._l1_frame(normalized_query, spt1)
            self._track_cost("L1", l1["cost"])
            framework = l1["framework"]
            spt2 = l1["spt2"]
            console.print("  ✓ Framework ready\n")

            # === L2: Gemini ↔ GPT Loop ===
            console.print("[yellow]→ L2: Collaborative Loop[/yellow]")
            l2 = await self._l2_loop(framework)
            total_cycles += l2["cycles"]
            converged_draft = l2["draft"]
            console.print(f"  ✓ Converged in {l2['cycles']} cycles\n")

            # === L3: SuperGrok Validation ===
            console.print("[yellow]→ L3: SuperGrok Validation[/yellow]")
            l3 = await self._l3_validate(converged_draft, spt1)
            self._track_cost("L3", l3["cost"])
            flagged_draft = l3["flagged_draft"]
            execution_flags = l3["execution_flags"]
            console.print(f"  ✓ {len(execution_flags)} code blocks flagged\n")

            # === L4: Claude Execute & Publish ===
            console.print("[yellow]→ L4: Claude Execute[/yellow]")
            l4 = await self._l4_execute(flagged_draft, spt2, execution_flags)
            self._track_cost("L4", l4["cost"])
            crm_score = l4["crm_score"]
            final_answer = l4["final_answer"]
            github_url = l4["github_url"]
            console.print(f"  ✓ CRM Score: {crm_score}/10")
            console.print(f"  ✓ GitHub: {github_url}\n")

            # === CRM Gate ===
            if crm_score <= self.config["pipeline"]["crm_abort_threshold"]:
                console.print("[bold red]⚠️ CRM ≤3: ABORT[/bold red]")
                return PipelineResult(
                    success=False,
                    query_id=self.query_id,
                    crm_score=crm_score,
                    abort_reason="Score too low",
                )

            # === L5: SuperGrok Repackage ===
            console.print("[yellow]→ L5: SuperGrok Repackage[/yellow]")
            l5 = await self._l5_repackage(github_url, final_answer, crm_score)
            self._track_cost("L5", l5["cost"])
            briefing = l5["briefing"]
            console.print("  ✓ Briefing ready\n")

            # === L7: SuperGrok Voice ===
            console.print("[yellow]→ L7: SuperGrok Voice[/yellow]")
            l7 = await self._l7_voice(briefing)
            self._track_cost("L7", l7["cost"])
            voice_script = l7["script"]
            console.print("  ✓ Voice ready\n")

            return PipelineResult(
                success=True,
                query_id=self.query_id,
                crm_score=crm_score,
                final_answer=final_answer,
                voice_script=voice_script,
                github_url=github_url,
                total_cost=sum(self.costs.values()),
                total_cycles=total_cycles,
            )

        except Exception as e:
            console.print(f"[bold red]❌ Failed: {e}[/bold red]")
            return PipelineResult(success=False, query_id=self.query_id, error=str(e))

    # Layer implementations (simplified for notebook)
    async def _l0_intake(self, voice_input: str) -> dict:
        # SuperGrok intake
        return {"normalized": voice_input, "spt1": {}, "cost": 0.001}

    async def _l1_frame(self, query: str, spt1: dict) -> dict:
        # Claude framing
        return {"framework": query, "spt2": {}, "cost": 0.01}

    async def _l2_loop(self, framework: str) -> dict:
        # Gemini ↔ GPT loop
        return {"draft": framework, "cycles": 3, "cost": 0.02}

    async def _l3_validate(self, draft: str, spt1: dict) -> dict:
        # SuperGrok validation
        return {"flagged_draft": draft, "execution_flags": [], "cost": 0.001}

    async def _l4_execute(self, draft: str, spt2: dict, flags: list) -> dict:
        # Claude execution
        return {
            "final_answer": draft,
            "github_url": f"https://github.com/{self.config['github']['repo_name']}",
            "crm_score": 8.0,
            "cost": 0.05,
        }

    async def _l5_repackage(self, url: str, answer: str, score: float) -> dict:
        # SuperGrok repackage
        return {"briefing": f"Research complete. Score: {score}/10", "cost": 0.001}

    async def _l7_voice(self, briefing: str) -> dict:
        # SuperGrok voice
        return {"script": briefing, "cost": 0.001}


print("UnGPTOrchestrator class loaded")

## Cell 4: Run Pipeline

In [ ]:
# Initialize orchestrator
orchestrator = UnGPTOrchestrator(CONFIG, API_KEYS)

# Test query
QUERY = "Analyze AAPL stock volatility for the past 30 days"

# Run pipeline
result = await orchestrator.run(QUERY)

# Display results
console.print("\n" + "=" * 50)
console.print(f"[cyan]Success:[/cyan] {result.success}")
console.print(f"[cyan]CRM Score:[/cyan] {result.crm_score}/10")
console.print(f"[cyan]Total Cost:[/cyan] ${result.total_cost:.4f}")
console.print(f"[cyan]Cycles:[/cyan] {result.total_cycles}")
console.print(f"[cyan]GitHub:[/cyan] {result.github_url}")
console.print("=" * 50)

## Cell 5: Voice Output

In [ ]:
# Display voice script
console.print("\n[bold]Voice Script:[/bold]")
console.print(result.voice_script)